# Tiny ears: an 8–10 MB bilingual sound recognizer (tiny-speech)

Trains a small model (about 7.5M parameters, **about 8.9 MB at 8 bits**) to hear the same Spanish and English sounds as a
large one (wav2vec2 espeak, 300M), and the sounds actually said in the Spanish transcripts, on free Mexican and Latin
American Spanish speech and some English. Code and details: `tiny-speech/training/ears/` in
github.com/larsboesen-prog/ranked-feed (branch `claude/tiny-speech-bilingual-v7lyp0`).

**MODE** (next cell): `"data"` on a CPU session (free on Kaggle: downloads and prepares the speech, about 2–3 hours), then
`"train"` on a GPU session with the data notebook's output added as input (labels, trains, exports: about 11 GPU hours of
Kaggle's 30 a week); `"smoke"` checks everything on a few clips in minutes; `"all"` does everything in one GPU session (Colab).
Kaggle: Settings → Accelerator (None for data and smoke, **GPU T4** for train), Internet **on**, then **Save Version →
Save & Run All**. Colab: Runtime → T4 GPU, MODE = "all", Run all (results in Google Drive, `MyDrive/tiny-ears`).

In [ ]:
MODE = "smoke"   # "smoke", "data" (CPU), "train" (GPU, needs the data notebook's output as input), "all"
import os, sys, shutil, pathlib, json, subprocess
KAGGLE = os.path.exists("/kaggle/working")
try:
    import google.colab  # noqa: F401
    COLAB = True
except ImportError:
    COLAB = False

if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = pathlib.Path("/content/drive/MyDrive/tiny-ears")   # kept between sessions: a new session resumes
    BASE = pathlib.Path("/content")
    HOURS = 11.0
elif KAGGLE:
    OUT = pathlib.Path("/kaggle/working/tiny-ears")           # the notebook's Output: what to keep (data mode: the packed speech)
    places = []
    for p in ("/kaggle/tmp", "/tmp"):
        try:
            os.makedirs(p, exist_ok=True)
            places.append(pathlib.Path(p))
        except OSError:
            pass
    BASE = max(places, key=lambda p: shutil.disk_usage(p).free)  # the roomiest scratch disk
    HOURS = {"smoke": 1.5, "data": 11.0, "train": 11.0, "all": 11.0}[MODE]  # Kaggle stops a session at 12 h (GPU: 30 h a week)
else:
    OUT, BASE, HOURS = pathlib.Path("tiny-ears").resolve(), pathlib.Path(".").resolve(), 11.0
WORK = BASE / "ears-work"                                     # audio and labels: large, thrown away at the end
REPO = BASE / "ranked-feed"
OUT.mkdir(parents=True, exist_ok=True)
DATA_FROM = None
if MODE == "train":  # the data notebook's output, added to this one as input: …/tiny-ears/data/manifest.jsonl
    found = sorted(pathlib.Path("/kaggle/input").rglob("data/manifest.jsonl")) if KAGGLE else sorted(OUT.rglob("data/manifest.jsonl"))
    assert found, "train mode: add the data notebook's output as input (Add Input → Your Work → tiny-ears-data)"
    DATA_FROM = found[0].parent
print({"mode": MODE, "kaggle": KAGGLE, "colab": COLAB, "out": str(OUT), "work": str(WORK), "data_from": str(DATA_FROM),
       "free_gb": round(shutil.disk_usage(BASE).free / 1e9, 1)})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# the code (public repository)
if not REPO.exists():
    !git clone -q --depth 1 --branch claude/tiny-speech-bilingual-v7lyp0 https://github.com/larsboesen-prog/ranked-feed.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log -1 --format="%h %s" | tee {OUT}/code-version.txt
EARS = REPO / "tiny-speech/training/ears"

In [ ]:
# everything for this MODE, in order; a finished stage is skipped when run again (resume after a stop)
extra = f"--data-from {DATA_FROM}" if DATA_FROM else ""
!{sys.executable} {EARS}/run.py --mode {MODE} --work {WORK} --out {OUT} --hours {HOURS} {extra}


In [ ]:
# what came out
report = OUT / "onnx/report.json"
if report.exists():
    print(report.read_text())
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in (".onnx", ".json", ".pt", ".jsonl"):
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(OUT)}")